import Pkg
Pkg.add("JuMP")
Pkg.add("HiGHS")

# Hot air - Airline

In [ ]:
# agiso@dtu.dk
using JuMP, HiGHS

cities = 4
time_periods = 3

D = zeros(Int, 3, 4, 4)

# period 1
D[1,:,:] = [
    0   50  53  14;
    84  0   80  21;
    17  58  0   40;
    31  79  34  0
]

# period 2
D[2,:,:] = [
    0   15  53  52;
    17  0   134 29;
    24  128 0   99;
    23  15  30  0
]

# period 3
D[3,:,:] = [
    0   3   16  9;
    48  0   104 48;
    62  92  0   68;
    13  15  21  0
]

P = [
    0    99   89   139;
    109  0    99   169;
    109  104  0    129;
    159  149  119  0
]

Ctakeoff = [
    0     5100  4400  8000;
    5100  0     11200 6900;
    4400  11200 0     5700;
    8000  6900  5700  0
]

hangers = [2, 1, 1, 0]
planes = 4

# Maximum number of passengers
M = 120

##### ----- Model ----- #####
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

##### ----- Variables ----- #####
@variable(model, x[1:time_periods, 1:cities, 1:cities] >= 0)
@variable(model, y[1:time_periods, 1:cities, 1:cities], Bin)

##### ----- objectives ----- #####
@objective(model, Max,
    sum(x[t, i, j] * P[i,j] for t in 1:time_periods, i in 1:1:cities, j in 1:1:cities) -
    sum(y[t, i, j] * Ctakeoff[i,j] for t in 1:time_periods, i in 1:1:cities, j in 1:1:cities)
)

##### ----- Constraints ----- #####
# The number of passengers should be lower than the demand
@constraint(model, [t in 1:time_periods, i in 1:1:cities, j in 1:1:cities],
    x[t,i,j] <= D[t,i,j]
)

# -----> BIG-M <-----
# Big-M link make y indicate wheter i plane is flown
@constraint(model, [t in 1:time_periods, i in 1:1:cities, j in 1:1:cities],
    x[t,i,j] <= M * y[t,i,j]
)


##### ----- Constraints - Keep track of aircraft ----- #####
# Overnight planes need to stay in specific hangers
@constraint(model, [i in 1:1:cities],
    sum(y[3,i,j] for j in 1:cities) == hangers[i]
)



#### ----- Optimize ----- #####
optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("\n Passengers")
for t in 1:3
    println("\nTime", t)
    for i in 1:4
        println(value.(x[t,i,:]))
    end
end

println("\n Planes in air")
for t in 1:3
    println("\nTime", t)
    for i in 1:4
        println(value.(y[t,i,:]))
    end
end


Optimal solution:
z = 22876.0

 Passengers

Time1
[0.0, 0.0, 53.0, 0.0]
[84.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0]
[0.0, 79.0, 0.0, 0.0]

Time2
[0.0, 0.0, 53.0, 0.0]
[0.0, 0.0, 120.0, 0.0]
[0.0, 120.0, 0.0, 99.0]
[0.0, 0.0, 0.0, 0.0]

Time3
[0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 48.0]
[0.0, 0.0, 0.0, 68.0]
[0.0, 0.0, 0.0, 0.0]

 Planes in air

Time1
[0.0, 0.0, 1.0, 0.0]
[1.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0]
[0.0, 1.0, 0.0, 0.0]

Time2
[0.0, 0.0, 1.0, 0.0]
[0.0, 0.0, 1.0, 0.0]
[0.0, 1.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0]

Time3
[0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0]
